In [ ]:
# check gpu
!nvidia-smi
import torch
print(torch.__version__, torch.version.cuda, torch.cuda.get_device_name(0), torch.cuda.is_bf16_supported())

In [ ]:
# mount drive
import os
from google.colab import drive
drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/BSS-AI"
os.makedirs(DRIVE_DIR, exist_ok=True)

In [ ]:
# clone repo
%cd /content
!test -d BSS-AI || git clone https://github.com/zi-wa/BSS-AI.git
%cd /content/BSS-AI

In [ ]:
# git pull
!git pull
!git log --oneline -1

In [ ]:
# install requirements
!grep -vxE "(torch|triton|adam-atan2) *" requirements.txt > /content/requirements_colab.txt
!pip install -q -r /content/requirements_colab.txt pytest
!pip install --no-cache-dir --no-build-isolation adam-atan2

In [ ]:
# wandb offline
os.environ["WANDB_MODE"] = "offline"
os.environ["WANDB_DIR"] = DRIVE_DIR

In [ ]:
# make tiny arc
import json
os.makedirs("data/tiny", exist_ok=True)

NUM_TRAIN , NUM_EVAL = 128, 16

for subset, num_puzzles in [("training", NUM_TRAIN), ("evaluation", NUM_EVAL)]:
    for kind in ["challenges", "solutions"]:
        with open(f"kaggle/combined/arc-agi_{subset}_{kind}.json") as f:
            puzzles = json.load(f)
        with open(f"data/tiny/arc-agi_{subset}_{kind}.json", "w") as f:
            json.dump({k: puzzles[k] for k in sorted(puzzles)[:num_puzzles]}, f)

In [ ]:
# build tiny dataset
NUM_AUG = 32
!python -m dataset.build_arc_dataset \
  --input-file-prefix data/tiny/arc-agi \
  --output-dir data/arc-tiny \
  --subsets training evaluation \
  --test-set-name evaluation \
  --num-aug $NUM_AUG

In [ ]:
# tiny configs
GLOBAL_BATCH_SIZE = 400 # 400 -> 72GB GPU RAM
EPOCHS = 5000
EVAL_INTERVAL = 500
LR_WARMUP_STEPS = 100
DATA_PATHS = "[data/arc-tiny]"

In [ ]:
# build full dataset
!python -m dataset.build_arc_dataset \
  --input-file-prefix kaggle/combined/arc-agi \
  --output-dir data/arc1concept-aug-1000 \
  --subsets training evaluation concept \
  --test-set-name evaluation

In [ ]:
# full configs
GLOBAL_BATCH_SIZE = 400
EPOCHS = 100000
EVAL_INTERVAL = 10000
LR_WARMUP_STEPS = 2000
DATA_PATHS = "[data/arc1concept-aug-1000]"

In [ ]:
# train trm
RUN_NAME = "trm"
!torchrun --nproc-per-node 1 --rdzv_backend=c10d --rdzv_endpoint=localhost:0 --nnodes=1 pretrain.py \
  arch=trm \
  data_paths=$DATA_PATHS \
  arch.L_layers=2 arch.H_cycles=3 arch.L_cycles=4 \
  global_batch_size=$GLOBAL_BATCH_SIZE \
  epochs=$EPOCHS \
  eval_interval=$EVAL_INTERVAL \
  lr_warmup_steps=$LR_WARMUP_STEPS \
  ema=True \
  +run_name={RUN_NAME} +checkpoint_path={DRIVE_DIR}/checkpoints/{RUN_NAME}

In [ ]:
# train uniform diffusion
RUN_NAME = "trm-uniform-diffusion"
!torchrun --nproc-per-node 1 --rdzv_backend=c10d --rdzv_endpoint=localhost:0 --nnodes=1 pretrain.py \
  arch=trm_diffusion \
  data_paths=$DATA_PATHS \
  arch.L_layers=2 arch.H_cycles=3 arch.L_cycles=4 \
  global_batch_size=$GLOBAL_BATCH_SIZE \
  epochs=$EPOCHS \
  eval_interval=$EVAL_INTERVAL \
  lr_warmup_steps=$LR_WARMUP_STEPS \
  ema=True \
  +run_name={RUN_NAME} +checkpoint_path={DRIVE_DIR}/checkpoints/{RUN_NAME}

In [ ]:
# train masked diffusion
RUN_NAME = "trm-masked-diffusion"
!torchrun --nproc-per-node 1 --rdzv_backend=c10d --rdzv_endpoint=localhost:0 --nnodes=1 pretrain.py \
  arch=trm_masked_diffusion \
  data_paths=$DATA_PATHS \
  arch.L_layers=2 arch.H_cycles=3 arch.L_cycles=4 \
  global_batch_size=$GLOBAL_BATCH_SIZE \
  epochs=$EPOCHS \
  eval_interval=$EVAL_INTERVAL \
  lr_warmup_steps=$LR_WARMUP_STEPS \
  ema=True \
  +run_name={RUN_NAME} +checkpoint_path={DRIVE_DIR}/checkpoints/{RUN_NAME}

In [ ]:
# reload and dump
import glob, re, time

ARCHES = {
    "trm": "arch=trm arch.L_layers=2 arch.H_cycles=3 arch.L_cycles=4",
    "trm-uniform-diffusion": "arch=trm_diffusion arch.L_layers=2 arch.H_cycles=3 arch.L_cycles=4",
    "trm-masked-diffusion": "arch=trm_masked_diffusion arch.L_layers=2 arch.H_cycles=3 arch.L_cycles=4",
}
RUN = "trm-uniform-diffusion"
ARCH = ARCHES[RUN]

ckpt = max((p for p in glob.glob(f"{DRIVE_DIR}/checkpoints/{RUN}/step_*") if re.search(r"step_\d+$", p)), key=lambda p: int(p.rsplit("_", 1)[1]))
RELOAD_DIR = f"{DRIVE_DIR}/checkpoints/{RUN}-reload-{time.strftime('%m%d-%H%M%S')}"
print(ckpt, "->", RELOAD_DIR)

!torchrun --nproc-per-node 1 --rdzv_backend=c10d --rdzv_endpoint=localhost:0 --nnodes=1 pretrain.py \
  {ARCH} data_paths=$DATA_PATHS \
  global_batch_size=$GLOBAL_BATCH_SIZE epochs=1 eval_interval=1 lr=0 puzzle_emb_lr=0 \
  +eval_save_outputs="[inputs,labels,preds,puzzle_identifiers]" \
  +load_checkpoint={ckpt} +run_name={RUN}-reload +checkpoint_path={RELOAD_DIR}

In [ ]:
# show model preds
import json, torch
import numpy as np, matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

data_dir = DATA_PATHS.strip("[]\"' ").split(",")[0]
names = json.load(open(f"{data_dir}/identifiers.json"))
path = max(glob.glob(f"{RELOAD_DIR}/step_*_all_preds.0"), key=lambda p: int(re.search(r"step_(\d+)_", p).group(1)))
saved = torch.load(path, map_location="cpu")

puzzle_ids = saved["puzzle_identifiers"].tolist()
rows = [i for i, pid in enumerate(puzzle_ids) if pid != 0 and "|||" not in names[pid]][:3]

# PAD 0, EOS 1, colors 2..11, ignored labels -100
palette = ['#000000', '#0074D9', '#FF4136', '#2ECC40', '#FFDC00', '#AAAAAA',
           '#F012BE', '#FF851B', '#7FDBFF', '#870C25', '#FFFFFF', '#555555']
cmap = ListedColormap(palette)

def to_display(tokens):
    grid = tokens.numpy().reshape(30, 30).astype(int)
    out = np.full_like(grid, 10)
    out[grid == 1] = 11
    colored = (grid >= 2) & (grid <= 11)
    out[colored] = grid[colored] - 2
    return out

fig, axes = plt.subplots(len(rows), 3, figsize=(9, 3 * len(rows)), squeeze=False)
for row, i in enumerate(rows):
    labels = saved["labels"][i]
    valid = labels != -100
    exact = bool((saved["preds"][i][valid] == labels[valid]).all())
    for col, key in enumerate(["inputs", "labels", "preds"]):
        title = f"{names[puzzle_ids[i]]} {key}" + (f" exact={exact}" if key == "preds" else "")
        axes[row][col].imshow(to_display(saved[key][i]), cmap=cmap, vmin=0, vmax=11)
        axes[row][col].set_title(title, fontsize=8)
        axes[row][col].axis("off")
plt.tight_layout(); plt.show()